# Interpretation

Этот ноутбук предназначен для глубокого анализа и интерпретации обученных моделей. Используется метод SHAP (SHapley Additive exPlanations) для понимания того, как признаки влияют на прогноз.

In [ ]:
import sys
import json
import joblib

import pandas as pd
import numpy as np

from pathlib import Path
import logging
from IPython.display import Image, display

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.evaluation.explainer import ModelExplainer

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
model_name = "logreg"  # logreg # xgb, lgbm, catboost

DATA_DIR = ROOT / "data"
PROCESSED_DATA = DATA_DIR / "processed"

TRAIN_DATA = PROCESSED_DATA / "train_features.parquet"
FEATURES_JSON = PROCESSED_DATA / "selected_features.json"

ARTIFACT_DIR = ROOT / "artifacts"
MODEL_DIR = ARTIFACT_DIR / f"{model_name}_model"

MODEL_PATH = MODEL_DIR / "model.joblib"
XAI_DIR = MODEL_DIR / "xai"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
XAI_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(TRAIN_DATA)

with open(FEATURES_JSON, "r") as f:
    selected_features = json.load(f)

X = df[selected_features].astype(np.float64)
y = df["TARGET"]

print(f"Dataset shape: {X.shape}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

model_wrapper = joblib.load(MODEL_PATH)
final_model = model_wrapper

print(f"Loaded model: {model_name}")
print(f"Model type: {type(final_model).__name__}")

Создадим SHAP Sample

In [ ]:
X_sample = final_model.transform(X.sample(min(1000, len(X)), random_state=42))
X_sample = final_model.transform(X_sample)
print(f"SHAP sample shape: {X_sample.shape}")

## Глобальная интерпретация (Global SHAP)
Помогает понять общие закономерности: какие признаки наиболее важны для модели и как их значения коррелируют с вероятностью дефолта.

#### **Beeswarm plot:**

Показывает, какие признаки наиболее влияют на вероятность дефолта. Цвет (красный/синий) означает значение признака, а положение на оси X — вклад в предсказание.

#### **Bar plot:**

Показывает глобальную важность признаков, основанную на средних абсолютных значениях SHAP.

In [ ]:
explainer = ModelExplainer(model=final_model, X_train=X_sample, feature_names=selected_features, output_dir=XAI_DIR)
logger.info("Explainer initialized successfully!")

explainer.compute_global_shap(X_sample=X_sample)

In [ ]:
display(Image(filename= XAI_DIR / "shap_summary_beeswarm.png"))
display(Image(filename= XAI_DIR / "shap_summary_bar.png"))

## Feature Importance

In [ ]:
importance_df = (explainer.get_feature_importance())

top_features = (explainer.get_feature_importance().head(3)["feature"].tolist())
print(f"Top features: {top_features}")

importance_df.head(20)

## Локальная интерпретация (Local SHAP)

#### **Waterfall plots**
Объясняют, почему модель приняла решение для конкретного клиента.

In [ ]:
probs = final_model.predict_proba(X_sample)

In [ ]:
# Клиент с высоким риском дефолта
high_risk_idx = np.argmax(probs)

print(f"HIGH RISK CLIENT (Prob: {probs[high_risk_idx]:.4f})")

instance_high = X_sample.iloc[high_risk_idx]

path_high = explainer.explain_local_shap(instance=instance_high, save_path= str (XAI_DIR / "local_shap_waterfall_high.png"))

display(Image(filename=path_high))

# Клиент с низким риском дефолта
low_risk_idx = np.argmin(probs)

print(f"LOW RISK CLIENT (Prob: {probs[low_risk_idx]:.4f})")

instance_low = X_sample.iloc[low_risk_idx]

path_low = explainer.explain_local_shap(instance=instance_low, save_path= str(XAI_DIR / f"local_shap_waterfall_low.png"))

display(Image(filename=path_low))


## Ключевые выводы
Интерпретация подтверждает, что модель принимает решения на основе логически значимых факторов риска.